# 03_描述統計
## Descriptive Statistics

This notebook summarises the two groups before running the two-proportion z-test.

- **Group variable:** `WhatIsYourSex` (1 = Female, 2 = Male)
- **Response variable:** `CurrentAlcoholUse_binary` (0 = non-user, 1 = current user)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

base_path = Path(__file__).resolve().parent.parent

input_path = base_path / "data" / "Processed" / "YRBS_2007_cleaned.csv"
output_fig = base_path / "outputs" / "figures"
output_tab = base_path / "outputs" / "tables"
output_fig.mkdir(parents=True, exist_ok=True)
output_tab.mkdir(parents=True, exist_ok=True)

# Debug
print("專案根目錄：", base_path)
print("CSV 路徑：", input_path)
print("檔案存在：", input_path.exists())

df = pd.read_csv(input_path)

group_col = "WhatIsYourSex"
response_col = "CurrentAlcoholUse_binary"
group_labels = {1: "Female", 2: "Male"}

df["Sex_label"] = df[group_col].map(group_labels)

print("Data shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head())

## Step 1: Group summary table

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

base_path = Path("..").resolve()

input_path = base_path / "data" / "Processed" / "YRBS_2007_cleaned.csv"
output_fig = base_path / "outputs" / "figures"
output_tab = base_path / "outputs" / "tables"
output_fig.mkdir(parents=True, exist_ok=True)
output_tab.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(input_path)

group_col = "WhatIsYourSex"
response_col = "CurrentAlcoholUse_binary"
group_labels = {1: "Female", 2: "Male"}

df["Sex_label"] = df[group_col].map(group_labels)

print("Data shape:", df.shape)
print("Columns:", df.columns.tolist())

# Group summary
summary = (
    df.groupby("Sex_label")[response_col]
    .agg(
        n="count",
        current_user_count="sum"
    )
    .reset_index()
)
summary["non_user_count"] = summary["n"] - summary["current_user_count"]
summary["proportion_current_user"] = (
    summary["current_user_count"] / summary["n"]
).round(4)

print("\nGroup summary table:")
print(summary[[
    "Sex_label", "n",
    "current_user_count",
    "non_user_count",
    "proportion_current_user"
]])

summary.to_csv(output_tab / "descriptive_summary.csv", index=False)
print("\nSaved to: ../outputs/tables/descriptive_summary.csv")

# Bar chart
fig, ax = plt.subplots(figsize=(6, 5))
colors = ["#f4a4a4", "#a4c4f4"]
bars = ax.bar(
    summary["Sex_label"],
    summary["proportion_current_user"],
    color=colors,
    edgecolor="black",
    width=0.5
)
for bar, val in zip(bars, summary["proportion_current_user"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{val:.4f}",
        ha="center", va="bottom", fontsize=11
    )
ax.set_title("Proportion of Current Alcohol Users by Sex", fontsize=13)
ax.set_xlabel("Sex", fontsize=11)
ax.set_ylabel("Proportion", fontsize=11)
ax.set_ylim(0, 1)
ax.axhline(y=0.5, color="gray", linestyle="--", linewidth=0.8, label="0.5 reference")
ax.legend()
plt.tight_layout()
plt.savefig(output_fig / "bar_proportion_by_sex.png", dpi=150)
plt.show()
print("Saved to: ../outputs/figures/bar_proportion_by_sex.png")

# Stacked bar chart
fig, ax = plt.subplots(figsize=(6, 5))
non_user_props = summary["non_user_count"] / summary["n"]
user_props = summary["proportion_current_user"]
ax.bar(summary["Sex_label"], non_user_props, color="#d9d9d9", edgecolor="black", label="Non-user (0)")
ax.bar(summary["Sex_label"], user_props, bottom=non_user_props, color="#a4c4f4", edgecolor="black", label="Current user (1)")
ax.set_title("Alcohol Use Distribution by Sex", fontsize=13)
ax.set_xlabel("Sex", fontsize=11)
ax.set_ylabel("Proportion", fontsize=11)
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.savefig(output_fig / "stacked_bar_by_sex.png", dpi=150)
plt.show()
print("Saved to: ../outputs/figures/stacked_bar_by_sex.png")

## Step 2: Bar chart — proportion of current alcohol users by sex

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

colors = ["#f4a4a4", "#a4c4f4"]
bars = ax.bar(
    summary["Sex_label"],
    summary["proportion_current_user"],
    color=colors,
    edgecolor="black",
    width=0.5
)

# Add value labels on bars
for bar, val in zip(bars, summary["proportion_current_user"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{val:.4f}",
        ha="center",
        va="bottom",
        fontsize=11
    )

ax.set_title("Proportion of Current Alcohol Users by Sex", fontsize=13)
ax.set_xlabel("Sex", fontsize=11)
ax.set_ylabel("Proportion", fontsize=11)
ax.set_ylim(0, 1)
ax.axhline(y=0.5, color="gray", linestyle="--", linewidth=0.8, label="0.5 reference")
ax.legend()

plt.tight_layout()
plt.savefig(output_fig / "bar_proportion_by_sex.png", dpi=150)
plt.show()
print("Saved to: ../outputs/figures/bar_proportion_by_sex.png")

## Step 3: Stacked bar chart — distribution within each group

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

non_user_props = summary["non_user_count"] / summary["n"]
user_props = summary["proportion_current_user"]

ax.bar(summary["Sex_label"], non_user_props, color="#d9d9d9", edgecolor="black", label="Non-user (0)")
ax.bar(summary["Sex_label"], user_props, bottom=non_user_props, color="#a4c4f4", edgecolor="black", label="Current user (1)")

ax.set_title("Alcohol Use Distribution by Sex", fontsize=13)
ax.set_xlabel("Sex", fontsize=11)
ax.set_ylabel("Proportion", fontsize=11)
ax.set_ylim(0, 1)
ax.legend()

plt.tight_layout()
plt.savefig(output_fig / "stacked_bar_by_sex.png", dpi=150)
plt.show()
print("Saved to: ../outputs/figures/stacked_bar_by_sex.png")

## Summary

| Sex | n | Current Users | Non-users | Proportion |
|-----|---|---------------|-----------|------------|
| Female | — | — | — | — |
| Male | — | — | — | — |

*(Numbers will be filled in after running the notebook.)*

Key observations:
- The two groups (Female vs Male) are compared on the proportion of current alcohol users.
- Descriptive results suggest whether there may be a difference before formal inference.
- The two-proportion z-test in `04_推論分析` will determine if the difference is statistically significant.

Saved outputs:
- `../outputs/tables/descriptive_summary.csv`
- `../outputs/figures/bar_proportion_by_sex.png`
- `../outputs/figures/stacked_bar_by_sex.png`